In [1]:
from typing import TypedDict, Annotated
import operator
import time
from langgraph.graph import StateGraph, START, END

# 1. Simple State
class StreamState(TypedDict):
    current_node: str
    counter: int
    log: Annotated[list, operator.add]



In [3]:
# 2. Nodes that simulate taking some time to process
def engine_room_node(state: StreamState):
    print("⚙️ Running Engine Node...")
    time.sleep(0.5) # Simulate workload lag
    return {"current_node": "engine", "counter": state["counter"] + 1, "log": ["Engine fired"]}

def bridge_node(state: StreamState):
    print("🛰️ Running Bridge Node...")
    time.sleep(0.5)
    return {"current_node": "bridge", "counter": state["counter"] + 1, "log": ["Bridge signaled"]}



In [4]:
# 3. Graph Assembly
builder = StateGraph(StreamState)
builder.add_node("engine", engine_room_node)
builder.add_node("bridge", bridge_node)

builder.add_edge(START, "engine")
builder.add_edge("engine", "bridge")
builder.add_edge("bridge", END)

streaming_app = builder.compile()



In [5]:
# Initial Data Packet
initial_payload = {"current_node": "start", "counter": 0, "log": ["System Boot"]}

# =========================================================
# DEMO 1: STREAM MODE = UPDATES
# =========================================================
print("=== DEMO 1: STREAMING NODE UPDATES ===")
# Notice we call .stream() instead of .invoke()
for chunk in streaming_app.stream(initial_payload, stream_mode="updates"):
    print(f"📡 Stream Yielded event: {chunk}")
    print("-" * 50)

# =========================================================
# DEMO 2: STREAM MODE = VALUES
# =========================================================
print("\n=== DEMO 2: STREAMING STATE VALUES ===")
for chunk in streaming_app.stream(initial_payload, stream_mode="values"):
    print(f"📡 Stream Yielded event: {chunk}")
    print("-" * 50)

=== DEMO 1: STREAMING NODE UPDATES ===
⚙️ Running Engine Node...
📡 Stream Yielded event: {'engine': {'current_node': 'engine', 'counter': 1, 'log': ['Engine fired']}}
--------------------------------------------------
🛰️ Running Bridge Node...
📡 Stream Yielded event: {'bridge': {'current_node': 'bridge', 'counter': 2, 'log': ['Bridge signaled']}}
--------------------------------------------------

=== DEMO 2: STREAMING STATE VALUES ===
📡 Stream Yielded event: {'current_node': 'start', 'counter': 0, 'log': ['System Boot']}
--------------------------------------------------
⚙️ Running Engine Node...
📡 Stream Yielded event: {'current_node': 'engine', 'counter': 1, 'log': ['System Boot', 'Engine fired']}
--------------------------------------------------
🛰️ Running Bridge Node...
📡 Stream Yielded event: {'current_node': 'bridge', 'counter': 2, 'log': ['System Boot', 'Engine fired', 'Bridge signaled']}
--------------------------------------------------
